# Social Media Aware Cleaning: Spell-checking and Link removals

In [1]:
!pip install spark-nlp==6.3.3 pyspark==3.3.1

In [2]:
# Import Spark NLP
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline
import sparknlp

from pyspark.sql.functions import regexp_extract

In [3]:
spark = sparknlp.start()

:: loading settings :: url = jar:file:/uufs/chpc.utah.edu/common/home/u1332544/my-python-venvs/spark-nlp/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /uufs/chpc.utah.edu/common/home/u1332544/.ivy2/cache
The jars for the packages stored in: /uufs/chpc.utah.edu/common/home/u1332544/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6bbb6f2c-e2e0-4097-8a50-b2efa209aee4;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.3.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in centr

26/03/26 00:04:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [9]:
training = spark.read.csv('data/twitter_training.csv', header=True)
col_names = ["id", "topic", "sentiment", "content"]
training = training.toDF(*col_names)
training.printSchema()

root
 |-- id: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- sentiment: string (nullable = true)
 |-- content: string (nullable = true)



Remove website links

In [12]:
website_regex = r"(http\w+)|(thesun\.\w+)|(\b\w+\.com\w+\b)|(\b\w+\.org\w+\b)|(\b\w+\.to\w+\b)|(\b\w+\.edu\w+\b)|(\b\w+\.net\w+\b)|(\b\w+\.gov\w+\b)|(\b\w+\.io\w+\b)|(\b\w+\.co\w+\b)|(\b\w+\.info\w+\b)|(\b\w+\.biz\w+\b)|(\b\w+\.me\w+\b)|(\b\w+\.tt\w+\b)|(\b\w+\.ly\w+\b)|(\b\w+\.it\w+\b)"
training = training.withColumn(
    "content",
    regexp_replace("content", website_regex, "")
)
print(training.show())
training = training.dropna()
training.printSchema()

+----+-----------+---------+--------------------+
|  id|      topic|sentiment|             content|
+----+-----------+---------+--------------------+
|2401|Borderlands| Positive|I am coming to th...|
|2401|Borderlands| Positive|im getting on bor...|
|2401|Borderlands| Positive|im coming on bord...|
|2401|Borderlands| Positive|im getting on bor...|
|2401|Borderlands| Positive|im getting into b...|
|2402|Borderlands| Positive|So I spent a few ...|
|2402|Borderlands| Positive|So I spent a coup...|
|2402|Borderlands| Positive|So I spent a few ...|
|2402|Borderlands| Positive|So I spent a few ...|
|2402|Borderlands| Positive|2010 So I spent a...|
|2402|Borderlands| Positive|                 was|
|2403|Borderlands|  Neutral|Rock-Hard La Varl...|
|2403|Borderlands|  Neutral|Rock-Hard La Varl...|
|2403|Borderlands|  Neutral|Rock-Hard La Varl...|
|2403|Borderlands|  Neutral|Rock-Hard La Vita...|
|2403|Borderlands|  Neutral|Live Rock - Hard ...|
|2403|Borderlands|  Neutral|I-Hard like me, R...|


In [20]:
document_assembler = DocumentAssembler()\
  .setInputCol("content")\
  .setOutputCol("document")

tokenizer = RecursiveTokenizer()\
  .setInputCols(["document"])\
  .setOutputCol("token")\
  .setPrefixes(["\"", "(", "[", "\n"])\
  .setSuffixes([".", ",", "?", ")","!", "‘s"])

normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized") \
    .setLowercase(False)

spell_checker = NorvigSweetingModel.pretrained() \
    .setInputCols(["normalized"]) \
    .setOutputCol("spell_checked")

finisher = Finisher().setInputCols("spell_checked").setOutputCols("cleaned_content").setOutputAsArray(False)

pipeline = Pipeline(stages = [document_assembler,
                              tokenizer,
                              normalizer,
                              spell_checker,
                              finisher])

spellcheck_norvig download started this may take some time.
26/03/26 00:12:19 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
Approximate size to download 4.2 MB
[OK!]


In [21]:
training_cleaned = pipeline.fit(training).transform(training)
print(training_cleaned.show())

+----+-----------+---------+--------------------+--------------------+
|  id|      topic|sentiment|             content|     cleaned_content|
+----+-----------+---------+--------------------+--------------------+
|2401|Borderlands| Positive|I am coming to th...|I@am@coming@to@th...|
|2401|Borderlands| Positive|im getting on bor...|im@getting@on@bor...|
|2401|Borderlands| Positive|im coming on bord...|im@coming@on@bord...|
|2401|Borderlands| Positive|im getting on bor...|im@getting@on@bor...|
|2401|Borderlands| Positive|im getting into b...|im@getting@into@b...|
|2402|Borderlands| Positive|So I spent a few ...|So@I@spent@a@few@...|
|2402|Borderlands| Positive|So I spent a coup...|So@I@spent@a@coup...|
|2402|Borderlands| Positive|So I spent a few ...|So@I@spent@a@few@...|
|2402|Borderlands| Positive|So I spent a few ...|So@I@spent@a@few@...|
|2402|Borderlands| Positive|2010 So I spent a...|So@I@spent@a@few@...|
|2402|Borderlands| Positive|                 was|                 was|
|2403|